# EDA — Hack Košice 2026 / UPJŠ Tear Challenge

Cieľ: pochopiť TRAIN_SET — štruktúra, triedy, formáty, rozmery, distribúcie.

In [ ]:
from pathlib import Path
import json, math, random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'TRAIN_SET'
print('Data root:', DATA, '— exists:', DATA.exists())

In [ ]:
# 1. Structure
dirs = sorted([d for d in DATA.iterdir() if d.is_dir()])
print(f'Top-level directories ({len(dirs)}):')
for d in dirs:
    print(' ', d.name)

In [ ]:
# 2. Per-class inventory
IMG_EXT = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}
rows = []
for d in dirs:
    files = list(d.rglob('*'))
    files = [f for f in files if f.is_file() and not f.name.startswith('.')]
    exts = Counter(f.suffix.lower() for f in files)
    imgs = [f for f in files if f.suffix.lower() in IMG_EXT]
    rows.append({
        'class': d.name,
        'n_files': len(files),
        'n_images': len(imgs),
        'extensions': dict(exts.most_common()),
        'total_MiB': sum(f.stat().st_size for f in files) / 1024 / 1024,
    })
df = pd.DataFrame(rows).sort_values('n_images', ascending=False)
df

In [ ]:
# 3. Class distribution bar plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(df['class'], df['n_images'])
ax.set_ylabel('# images')
ax.set_title('Samples per class')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 4. Inspect dimensions / dtype across 20 random images per class
random.seed(0)
probe = []
for d in dirs:
    imgs = [f for f in d.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXT]
    for f in random.sample(imgs, min(20, len(imgs))):
        try:
            with Image.open(f) as im:
                arr = np.array(im)
            probe.append({
                'class': d.name, 'file': f.name,
                'W': im.size[0], 'H': im.size[1],
                'mode': im.mode, 'dtype': str(arr.dtype),
                'min': float(arr.min()), 'max': float(arr.max()),
                'mean': float(arr.mean()),
            })
        except Exception as e:
            probe.append({'class': d.name, 'file': f.name, 'error': str(e)[:80]})
probe_df = pd.DataFrame(probe)
probe_df.head(20)

In [ ]:
# 5. Resolution summary
if 'W' in probe_df.columns:
    print(probe_df[['W','H']].describe())
    print('\nUnique (W,H) combinations:')
    print(probe_df.groupby(['W','H']).size().sort_values(ascending=False).head(20))

In [ ]:
# 6. Visual grid — 4 random samples per class
n_cls = len(dirs)
fig, axes = plt.subplots(n_cls, 4, figsize=(14, 3.5*n_cls))
for i, d in enumerate(dirs):
    imgs = [f for f in d.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXT]
    pick = random.sample(imgs, min(4, len(imgs)))
    for j, ax in enumerate(axes[i]):
        if j < len(pick):
            with Image.open(pick[j]) as im:
                arr = np.array(im)
            cmap = None if (arr.ndim == 3 and arr.shape[-1] in (3,4)) else 'afmhot'
            ax.imshow(arr, cmap=cmap)
            ax.set_title(pick[j].name, fontsize=7)
        ax.axis('off')
    axes[i,0].set_ylabel(d.name, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Height histogram per class (reveals scale structure differences)
fig, axes = plt.subplots(math.ceil(n_cls/3), 3, figsize=(14, 3*math.ceil(n_cls/3)))
axes = axes.flatten()
for i, d in enumerate(dirs):
    imgs = [f for f in d.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXT]
    sample = random.sample(imgs, min(10, len(imgs)))
    vals = []
    for f in sample:
        with Image.open(f) as im:
            a = np.array(im).astype(float).flatten()
            if a.size > 20000:
                a = np.random.choice(a, 20000, replace=False)
            vals.append(a)
    axes[i].hist(np.concatenate(vals), bins=100, color='C0', alpha=0.7)
    axes[i].set_title(d.name, fontsize=9)
plt.tight_layout()
plt.show()